# Lecture 4: Spectral Analysis
- 4.1 Spectral Densities
- 4.2 The Periodogram
- 4.3 Time-Invariant Linear Filters
- 4.4 Spectral Density of ARMA Processes


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq
import warnings; warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 4)
print("Ready!")


## 4.1 Spectral Densities

The **spectral density function** (power spectral density) of a stationary process:
$$f(\lambda) = \frac{1}{2\pi} \sum_{h=-\infty}^{\infty} \gamma(h) e^{-ih\lambda}, \quad \lambda \in [-\pi, \pi]$$

**Inverse transform (Herglotz theorem):**
$$\gamma(h) = \int_{-\pi}^{\pi} e^{ih\lambda} f(\lambda) d\lambda$$

In particular: $\gamma(0) = \text{Var}(X_t) = \int_{-\pi}^{\pi} f(\lambda) d\lambda$

Interpretation: $f(\lambda)\Delta\lambda$ ≈ variance contribution from the frequency band $[\lambda, \lambda+\Delta\lambda]$


In [ ]:
# ── Theoretical Spectral Density Computation ──

def theoretical_spectral_density(ar_params, ma_params, sigma2=1.0, n_freq=500):
    '''
    Theoretical spectral density for ARMA(p,q):
    f(lambda) = (sigma^2 / 2pi) |theta(e^{i*lambda})|^2 / |phi(e^{i*lambda})|^2
    '''
    lambdas = np.linspace(-np.pi, np.pi, n_freq)
    f = np.zeros(n_freq)
    
    for k, lam in enumerate(lambdas):
        z = np.exp(1j * lam)
        
        # AR polynomial: phi(z) = 1 - phi1*z - phi2*z^2 - ...
        ar_poly = 1.0
        for j, phi in enumerate(ar_params):
            ar_poly -= phi * z**(j+1)
        
        # MA polynomial: theta(z) = 1 + theta1*z + theta2*z^2 + ...
        ma_poly = 1.0
        for j, theta in enumerate(ma_params):
            ma_poly += theta * z**(j+1)
        
        f[k] = (sigma2 / (2 * np.pi)) * np.abs(ma_poly)**2 / np.abs(ar_poly)**2
    
    return lambdas, f

# Different models
models = {
    'White Noise':      ([], [], 'gray'),
    'AR(1) phi=0.9':    ([0.9], [], 'steelblue'),
    'AR(1) phi=-0.9':   ([-0.9], [], 'crimson'),
    'AR(2) complex':    ([1.0, -0.5], [], 'green'),
    'MA(1) theta=0.8':  ([], [0.8], 'orange'),
    'ARMA(1,1)':        ([0.7], [0.4], 'purple'),
}

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, (name, (ar, ma, color)) in enumerate(models.items()):
    lam, f = theoretical_spectral_density(ar, ma)
    
    axes[idx].plot(lam, f, color=color, lw=2)
    axes[idx].fill_between(lam, 0, f, alpha=0.15, color=color)
    axes[idx].set_title(f'{name}', fontsize=11)
    axes[idx].set_xlabel('Frequency lambda')
    axes[idx].set_ylabel('f(lambda)')
    axes[idx].set_xlim(-np.pi, np.pi)
    axes[idx].set_xticks([-np.pi, -np.pi/2, 0, np.pi/2, np.pi])
    axes[idx].set_xticklabels(['-pi', '-pi/2', '0', 'pi/2', 'pi'])
    
    # Total variance check
    var_approx = np.trapezoid(f, lam)
    axes[idx].set_xlabel(f'Frequency lambda  [int f dlambda = {var_approx:.3f} ~ gamma(0)]')

plt.tight_layout()
plt.show()
print("Interpretations:")
print("  AR phi>0: high power at low frequencies (slow fluctuations)")
print("  AR phi<0: high power at high frequencies (rapid fluctuations)")
print("  White noise: equal power at all frequencies (flat spectrum)")


## 4.2 The Periodogram

The **periodogram** as a sample spectrum estimator:
$$I(\lambda_j) = \frac{1}{n}\left|\sum_{t=1}^{n} X_t e^{-it\lambda_j}\right|^2, \quad \lambda_j = \frac{2\pi j}{n}$$

$E[I(\lambda_j)] \approx f(\lambda_j)$, but smoothing is needed due to high variance.

Periodogram values are **asymptotically independent** with distribution $\text{Exp}(f(\lambda_j))$.


In [ ]:
# ── Periodogram Computation and Interpretation ──

def periodogram(x):
    n = len(x)
    # FFT-based computation
    Xf = fft(x - x.mean())
    I = (np.abs(Xf)**2) / n
    freqs = fftfreq(n) * 2 * np.pi  # [0, 2pi] → [-pi, pi]
    # Positive frequencies only
    pos_idx = freqs > 0
    return freqs[pos_idx], I[pos_idx]

np.random.seed(42)
n = 256

# Three different processes
processes = {
    'Signal (f=0.1) + Noise': np.cos(2*np.pi*0.1*np.arange(n)) + np.random.normal(0,0.5,n),
    'AR(1) phi=0.8': None,
    'White Noise': np.random.normal(0, 1, n),
}

from statsmodels.tsa.arima_process import arma_generate_sample
processes['AR(1) phi=0.8'] = arma_generate_sample([1,-0.8],[1], n)

fig, axes = plt.subplots(len(processes), 2, figsize=(14, 10))

for i, (name, X) in enumerate(processes.items()):
    # Left: Time series
    axes[i, 0].plot(X, lw=0.8, color='steelblue')
    axes[i, 0].set_title(f'{name} — Time Series')
    
    # Right: Periodogram
    freqs, I = periodogram(X)
    axes[i, 1].plot(freqs/np.pi, I, lw=0.7, color='steelblue', alpha=0.7)
    axes[i, 1].set_title(f'{name} — Periodogram')
    axes[i, 1].set_xlabel('Frequency (lambda/pi)')
    axes[i, 1].set_ylabel('I(lambda)')
    
    # Peak
    peak_idx = np.argmax(I)
    axes[i, 1].axvline(freqs[peak_idx]/np.pi, color='red', ls='--', lw=1.5,
                        label=f'Peak: lambda={freqs[peak_idx]/np.pi:.3f}*pi')
    axes[i, 1].legend(fontsize=9)

plt.tight_layout()
plt.show()

# Detecting the sinusoidal signal
X_signal = np.cos(2*np.pi*0.1*np.arange(n)) + np.random.normal(0, 0.5, n)
freqs, I = periodogram(X_signal)
peak_freq = freqs[np.argmax(I)] / (2 * np.pi)
print(f"Detected dominant frequency: {peak_freq:.4f} Hz")
print(f"Expected:                     0.1000 Hz")
print(f"Period: {1/peak_freq:.1f} samples")


In [ ]:
# ── Smoothed Periodogram (Spectral Estimation) ──
from scipy.signal import welch, periodogram as sp_periodogram

np.random.seed(10)
n = 512
X = arma_generate_sample([1, -0.8], [1], n)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1) Raw periodogram
freqs_raw, I_raw = sp_periodogram(X, scaling='density')
axes[0].semilogy(freqs_raw * np.pi * 2, I_raw, lw=0.5, color='steelblue')
axes[0].set_title('Raw Periodogram (high variance)')
axes[0].set_xlabel('Frequency'); axes[0].set_ylabel('PSD')

# 2) Welch method (overlapping windows)
for nfft_win in [32, 64, 128]:
    freqs_w, Pxx_w = welch(X, nperseg=nfft_win, scaling='density')
    axes[1].semilogy(freqs_w * np.pi * 2, Pxx_w, lw=1.5, 
                      label=f'Window={nfft_win}', alpha=0.8)
axes[1].set_title('Welch Method (smoothed)')
axes[1].set_xlabel('Frequency'); axes[1].legend(fontsize=9)

# 3) Theoretical vs Welch
lam_th, f_th = theoretical_spectral_density([0.8], [])
freqs_w, Pxx_w = welch(X, nperseg=64, scaling='density')
axes[2].plot(lam_th[lam_th>=0], f_th[lam_th>=0], 'r-', lw=2, label='Theoretical f(lambda)')
axes[2].semilogy(freqs_w * np.pi * 2, Pxx_w, 'b-', lw=1.5, label='Welch estimate')
axes[2].set_title('Theoretical vs Welch Estimate'); axes[2].legend()
axes[2].set_xlabel('Frequency [0, pi]')

plt.tight_layout(); plt.show()


## 4.3 Time-Invariant Linear Filters

**Linear filter:** $Y_t = \sum_{j=-\infty}^{\infty} a_j X_{t-j}$

**Transfer function:** $A(\lambda) = \sum_j a_j e^{-ij\lambda}$

Spectral density of the filtered process:
$$f_Y(\lambda) = |A(\lambda)|^2 f_X(\lambda)$$

Examples:
- **Low-pass filter:** attenuates high frequencies
- **High-pass filter:** attenuates low frequencies


In [ ]:
# ── Linear Filters: Low-pass / High-pass ──
from scipy.signal import butter, filtfilt

np.random.seed(5)
n = 300
t = np.arange(n)

# Mixed signal: slow trend + fast oscillation + noise
slow = 2 * np.sin(2 * np.pi * t / 50)     # Low-frequency (trend)
fast = 0.5 * np.sin(2 * np.pi * t / 5)    # High-frequency (noise-like)
noise = np.random.normal(0, 0.3, n)
X = slow + fast + noise

# Butterworth filters
def butter_lowpass(cutoff, fs=1.0, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype='low')
    return b, a

def butter_highpass(cutoff, fs=1.0, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype='high')
    return b, a

b_low, a_low = butter_lowpass(0.1)
b_high, a_high = butter_highpass(0.1)

X_low  = filtfilt(b_low, a_low, X)
X_high = filtfilt(b_high, a_high, X)

fig, axes = plt.subplots(4, 1, figsize=(13, 12), sharex=True)
axes[0].plot(X, 'k', lw=0.8, label='Original X(t)')
axes[0].set_title('Original Series = Slow + Fast + Noise')
axes[1].plot(slow, 'r--', lw=1.5, label='True slow component'); axes[1].plot(X_low, 'r', lw=1.5, label='LP filter output')
axes[1].set_title('Low-pass Filter (< 0.1 Hz)'); axes[1].legend()
axes[2].plot(fast+noise, 'b--', lw=1, label='True fast+noise'); axes[2].plot(X_high, 'b', lw=1.5, label='HP filter output')
axes[2].set_title('High-pass Filter (> 0.1 Hz)'); axes[2].legend()

# Spectral comparison
freqs, I_orig  = periodogram(X)
_, I_low   = periodogram(X_low)
_, I_high  = periodogram(X_high)
axes[3].plot(freqs/np.pi, I_orig, 'k', lw=0.7, label='Original', alpha=0.6)
axes[3].plot(freqs/np.pi, I_low, 'r', lw=1.5, label='LP')
axes[3].plot(freqs/np.pi, I_high, 'b', lw=1.5, label='HP')
axes[3].axvline(0.2, color='gray', ls='--', label='Cutoff frequency')
axes[3].set_xlabel('Frequency (lambda/pi)'); axes[3].set_ylabel('Power'); axes[3].legend()
axes[3].set_title('Spectral Analysis: Filter Effect')
plt.tight_layout(); plt.show()


## 4.4 Spectral Density of ARMA Processes

**For ARMA(p,q):**
$$f(\lambda) = \frac{\sigma^2}{2\pi} \frac{|\theta(e^{i\lambda})|^2}{|\phi(e^{i\lambda})|^2}$$

**Special cases:**
- AR(1): $f(\lambda) = \frac{\sigma^2}{2\pi(1 + \phi^2 - 2\phi\cos\lambda)}$
- MA(1): $f(\lambda) = \frac{\sigma^2}{2\pi}(1 + \theta^2 + 2\theta\cos\lambda)$


In [ ]:
# ── ARMA Spectral Density: Theory vs Periodogram ──
np.random.seed(99)
n = 1024  # Large n => better estimate

model_params = [
    ('AR(1) phi=0.8',   [0.8],    []),
    ('AR(2)',            [1.0,-0.5],[]),
    ('MA(1) theta=0.7', [],       [0.7]),
    ('ARMA(1,1)',        [0.7],    [0.5]),
]

fig, axes = plt.subplots(1, len(model_params), figsize=(16, 4))

for idx, (name, ar, ma) in enumerate(model_params):
    # Simulate
    ar_poly = np.array([1] + [-p for p in ar])
    ma_poly = np.array([1] + list(ma))
    X = arma_generate_sample(ar_poly, ma_poly, n, scale=1)
    
    # Smoothed periodogram
    freqs_w, Pxx = welch(X, nperseg=128, scaling='density')
    
    # Theoretical spectrum
    lam_th, f_th = theoretical_spectral_density(ar, ma, sigma2=1.0)
    
    axes[idx].plot(lam_th[lam_th >= 0], f_th[lam_th >= 0], 
                    'r-', lw=2.5, label='Theoretical f(lambda)', alpha=0.8)
    axes[idx].plot(freqs_w * 2 * np.pi, Pxx / (2*np.pi), 
                    'b-', lw=1, label='Welch estimate', alpha=0.7)
    axes[idx].set_title(name, fontsize=10)
    axes[idx].set_xlabel('Frequency lambda in [0, pi]')
    axes[idx].legend(fontsize=8)
    axes[idx].set_xlim(0, np.pi)

axes[0].set_ylabel('Spectral Density f(lambda)')
plt.tight_layout(); plt.show()

# Spectral peak analysis
print("Spectral peak analysis:")
for name, ar, ma in model_params:
    lam_th, f_th = theoretical_spectral_density(ar, ma)
    peak_lam = lam_th[np.argmax(f_th)]
    print(f"  {name:<20}: Peak lambda = {peak_lam:.4f} rad ({peak_lam/np.pi:.3f}*pi)")


## Summary: Spectral Analysis

| Concept | Definition | Use |
|---------|-----------|-----|
| **Spectral Density** | $f(\lambda) = \frac{1}{2\pi}\sum \gamma(h)e^{-ih\lambda}$ | Theoretical power distribution |
| **Periodogram** | $I(\lambda_j) = \frac{1}{n}\|DFT(X)\|^2$ | Raw spectrum estimate |
| **Welch Method** | Average over overlapping windows | Smoothed estimate |
| **LP/HP Filter** | $f_Y = |A(\lambda)|^2 f_X$ | Component separation |
| **ARMA Spectrum** | $f = \frac{\sigma^2}{2\pi}\frac{|\theta(e^{i\lambda})|^2}{|\phi(e^{i\lambda})|^2}$ | Model-based estimate |

